In [36]:
import os
import pandas as pd
import numpy as np
from numba import njit, float64, int64, uint64,types
from numba.typed import Dict

In [37]:

# 현재 파일들이 있는 그 위치 그대로 설정
Base_dir = "C:/Users/user/Desktop/IDS_masters/9) Car-Hacking Dataset"

# 파일 이름에 포함된 단어로 공격 유형 구분
attack_mapping = {
    "Dos": 1,
    "Fuzzing": 2,
    "Spoofing":4
}

attack_files = []

# 폴더 안을 바로 검사
for attack_name, attack_id in attack_mapping.items():
    attack_dir = os.path.join(Base_dir, attack_name)
    if not os.path.isdir(attack_dir):
        continue

    for fname in os.listdir(attack_dir):
        if fname.endswith(".csv"):
            attack_files.append({
                "path": os.path.join(attack_dir, fname),
                "attack_id": attack_id
            })

In [38]:
#################################
# 2. Visualization Mirgu Dataset
#################################
hash_cache = {}

# [ADD] payload 8바이트 리스트로 만드는 함수 (너가 쓰던 스타일)
def parse_payload(row):
    # row에는 b0~b7 컬럼이 있고, 이미 0패딩되어 있음
    return [int(row[f"b{i}"]) for i in range(8)]

def process_csv_file(path, attack_id):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split(",")
            if len(parts) < 4:
                continue

            ts_str, canid_raw, dlc_str = parts[0], parts[1], parts[2]
            label = parts[-1].strip()         # [MINOR] strip
            data_tokens = parts[3:-1]

            # dlc/ts 파싱
            try:
                ts = float(ts_str)
                dlc = int(dlc_str)
            except:
                continue

            # payload bytes: DLC 만큼만 읽고, 8바이트로 0 패딩
            payload = []
            for i in range(min(dlc, len(data_tokens), 8)):
                tok = data_tokens[i].strip()
                if tok == "" or tok.lower() == "nan":
                    payload.append(0)
                else:
                    try:
                        payload.append(int(tok, 16))
                    except:
                        payload.append(0)

            payload += [0] * (8 - len(payload))
            payload = payload[:8]

            rows.append([ts, canid_raw, dlc, *payload, label])

    df = pd.DataFrame(
        rows,
        columns=["timestamp", "CAN_ID", "DLC"] + [f"b{i}" for i in range(8)] + ["Label"]
    )

    # CAN_ID int 변환
    df["int_CAN_ID"] = df["CAN_ID"].apply(lambda x: int(str(x).strip(), 16)).astype(np.int64)


    # Payloads 컬럼 추가 
    df["Payloads"] = df.apply(parse_payload, axis=1).tolist()

    # 라벨링
    df["Labeling"] = df["Label"].map({"T": attack_id, "R": 0}).fillna(0).astype(int)

    df = df[["timestamp","CAN_ID","int_CAN_ID","Payloads","Labeling"]]

    return df


In [39]:
@njit
def popcount64(x):
    c = 0
    v = int64(x)
    while v:
        v &= v - int64(1)
        c += 1
    return c

@njit
def pack_payload_u64(row):
    v = uint64(0)
    for i in range(8):
        v |= uint64(row[i]) << (i * 8)
    return v

@njit
def update_ema_z(val, cid, ema_map, sq_ema_map, alpha):
    if cid not in ema_map:
        ema_map[cid] = float64(val)
        sq_ema_map[cid] = float64(val ** 2)
        return 0.0
    mean = ema_map[cid]
    sq_mean = sq_ema_map[cid]
    var = sq_mean - (mean ** 2)
    if var < 0: var = 0.0
    std = np.sqrt(var)
    z = 0.0
    if std > 1e-9:
        z = (val - mean) / std
        if z > 5.0: z = 5.0
        elif z < -5.0: z = -5.0
    ema_map[cid] = (1.0 - alpha) * mean + alpha * val
    sq_ema_map[cid] = (1.0 - alpha) * sq_mean + alpha * (val ** 2)
    return z

@njit(fastmath=True)
def calculate_features_15_numba(timestamps, can_ids, payloads):
    n = len(timestamps)
    features = np.zeros((n, 15), dtype=np.float64)
    
    # 상태 관리
    last_time_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_payload_map = Dict.empty(key_type=types.int64, value_type=types.uint64)
    last_id_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_iat_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_global_index_map = Dict.empty(key_type=types.int64, value_type=types.int64)
    id_ham_ema = Dict.empty(key_type=types.int64, value_type=types.float64)
    
    # Anchor & Warm-up (128개 윈도우 고려)
    WARM_UP_LIMIT = 4000 
    anchor_iat_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    anchor_gap_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_normal_time_map = Dict.empty(key_type=types.int64, value_type=types.float64)

    # Z-Score & Global
    ema_freq = Dict.empty(key_type=types.int64, value_type=types.float64)
    sq_ema_freq = Dict.empty(key_type=types.int64, value_type=types.float64)
    ema_jit = Dict.empty(key_type=types.int64, value_type=types.float64)
    sq_ema_jit = Dict.empty(key_type=types.int64, value_type=types.float64)
    ema_global = Dict.empty(key_type=types.int64, value_type=types.float64)
    sq_ema_global = Dict.empty(key_type=types.int64, value_type=types.float64)

    G_KEY = np.int64(-1)
    alpha_slow = 0.001
    alpha_ham = 0.05
    eps = 1e-9
    prev_global_time = timestamps[0]

    for i in range(n):
        # 128개 윈도우 기준 로컬 빈도 초기화
        if (i % 128) == 0:
            last_id_map.clear()
            
        ts = timestamps[i]
        cid = can_ids[i]
        row = payloads[i]
        if np.isnan(ts): ts = prev_global_time

        # 1. 물리량
        curr_iat = max(0.0, ts - last_time_map[cid]) if cid in last_time_map else 0.001
        curr_packet_gap = float64(i - last_global_index_map[cid]) if cid in last_global_index_map else 100.0
        curr_freq = 1.0 / (curr_iat + eps)
        curr_jit = np.abs(curr_iat - last_iat_map[cid]) if cid in last_iat_map else 0.0

       # 2. [핵심] Warm-up: 시간 주기와 패킷 개수 주기를 동시에 학습/고정
        if i < WARM_UP_LIMIT:
            if cid not in anchor_iat_map:
                anchor_iat_map[cid] = curr_iat
                anchor_gap_map[cid] = curr_packet_gap
            else:
                # 4000개까지는 부드럽게 평균을 쌓음
                anchor_iat_map[cid] = 0.99 * anchor_iat_map[cid] + 0.01 * curr_iat
                anchor_gap_map[cid] = 0.99 * anchor_gap_map[cid] + 0.01 * curr_packet_gap
                
        # 3. 시간/패킷 기반 스푸핑 분석 (Clipping)
        iat_ratio,packet_gap_ratio, phase_offset = 1.0,1.0, 0.0
        if cid in anchor_iat_map:
            # 3-1. 시간 기반 분석
            base_iat = anchor_iat_map[cid]
            if base_iat > 1e-7:
                iat_ratio = min(curr_iat / base_iat, 2.0)
                diff = ts - (last_normal_time_map[cid] + base_iat) if cid in last_normal_time_map else 0.0
                phase_offset = max(min(diff / base_iat, 1.0), -1.0)
            
            # 3-2. [신규] 패킷 개수 기반 분석 (사용자 제안 로직)
            base_gap = anchor_gap_map[cid]
            if base_gap > 0.5:
                # 평소 100개 뒤에 나오던 게 5개 뒤에 나오면 0.05가 됨
                packet_gap_ratio = min(curr_packet_gap / base_gap, 2.0)

            # 리듬 업데이트 (시간 기준)
            if abs(phase_offset) < 0.2:
                last_normal_time_map[cid] = ts
        else:
            last_normal_time_map[cid] = ts

        # 4. 데이터 내용 기반 (Hamming & Entropy)
        cur_bytes = pack_payload_u64(row)
        rel_change = 0.0
        if cid in last_payload_map:
            h_dist = float64(popcount64(cur_bytes ^ last_payload_map[cid]))
            avg_h = id_ham_ema.get(cid, h_dist)
            rel_change = h_dist / (avg_h + 0.1)
            id_ham_ema[cid] = (1.0 - alpha_ham) * avg_h + alpha_ham * h_dist
        last_payload_map[cid] = cur_bytes

        # Payload Entropy
        p_counts = np.zeros(256, dtype=np.int64)
        for b in row: p_counts[b] += 1
        ent = 0.0
        for c in p_counts:
            if c > 0:
                p = c / 8.0
                ent -= p * np.log(p)

        # Diff Entropy
        d_counts = Dict.empty(key_type=types.int64, value_type=types.float64)
        for b_idx in range(7):
            d = (int64(row[b_idx+1]) - int64(row[b_idx])) % 256
            d_counts[d] = d_counts.get(d, 0.0) + 1.0
        d_ent = 0.0
        for dv in d_counts:
            pk = d_counts[dv] / 7.0
            d_ent -= pk * np.log(pk + 1e-9)

        # 윈도우 ID 엔트로피 (최근 128개 패킷 대상)
        wi_ent = 0.0
        if i >= 127:
            win_id_counts = Dict.empty(key_type=types.int64, value_type=types.float64)
            for k in range(i-127, i+1):
                wid = can_ids[k]
                win_id_counts[wid] = win_id_counts.get(wid, 0.0) + 1.0
            for wid in win_id_counts:
                pk = win_id_counts[wid] / 128.0
                wi_ent -= pk * np.log(pk + 1e-9)

        # 5. 피처 할당
        features[i, 0] = np.log1p(curr_iat * 1000.0) / 7.0
        features[i, 1] = 1.0 if cid == 0 else 0.0
        features[i, 2] = ent / 2.1
        features[i, 3] = np.log1p(ent * rel_change)
        features[i, 4] = np.log1p(rel_change / (curr_iat + eps)) / 10.0
        cnt = last_id_map.get(cid, 0.0) + 1.0
        last_id_map[cid] = cnt
        features[i, 5] = cnt / 128.0
        features[i, 6] = np.log1p(rel_change) / 5.0
        features[i, 7] = d_ent / 1.94
        features[i, 8] = wi_ent / 4.85
        features[i, 9] = update_ema_z(curr_freq, cid, ema_freq, sq_ema_freq, alpha_slow)
        features[i, 10] = update_ema_z(curr_iat, cid, ema_jit, sq_ema_jit, alpha_slow)
        features[i, 11] = update_ema_z(curr_freq, G_KEY, ema_global, sq_ema_global, alpha_slow)
        features[i, 12] = iat_ratio
        features[i, 13] = phase_offset
        features[i, 14] = packet_gap_ratio

        last_time_map[cid] = ts
        last_iat_map[cid] = curr_iat
        last_global_index_map[cid] = i
        prev_global_time = ts

    return features

In [40]:
# ==========================================
# 4. Making Feature with Numba
# ==========================================

def Make_feature(path, attack_id):

    df = process_csv_file(path, attack_id)

    # ======== to numpy ========== #
    timestamps = df["timestamp"].to_numpy(np.float32)
    can_ids = df["int_CAN_ID"].to_numpy(np.int64)
    payloads = np.array(df["Payloads"].tolist(), dtype=np.uint8)
    labels = df["Labeling"].to_numpy(np.int64)

    
    # ======== calculate feature ========== #
    feature9 = calculate_features_15_numba(timestamps, can_ids, payloads)
    print(feature9.shape)

    return feature9, labels

In [41]:
# ==========================================
# 5. Slide Window and Label
# ==========================================
def Sliding_Window_and_Labeling(feature, label, win_size=128, stride=64):
    windows = []
    labels = []
    n = feature.shape[0]
    for start in range(0, n-win_size+1 , stride):
        end = start + win_size
        windows.append(feature[start:end])
        labels.append(label[start:end])


    return (
        np.stack(windows, axis=0).astype(np.float32),
        np.stack(labels, axis=0).astype(np.int64)
    )

In [42]:
# ==========================================
# 6. main
# ==========================================
all_x = []
all_y = []

for item in attack_files:
    feature9, labels = Make_feature(item["path"], item["attack_id"]) # 각 feature 추출
    windows, y = Sliding_Window_and_Labeling(feature9,labels) # 윈도우 만들기

    all_x.append(windows)
    all_y.append(y)

all_x_win = np.concatenate(all_x, axis=0)
all_y_win = np.concatenate(all_y, axis=0)

(3665771, 15)
(3838860, 15)
(4443142, 15)
(4621702, 15)


In [43]:
# ==========================================
# 7. Save
# ==========================================
import numpy as np
np.savez(
    "C:/Users/user/Desktop/IDS_masters/dataset/carhacking_test_0212_816.npz",
    X = all_x_win.astype(np.float32),
    y = all_y_win.astype(np.int64)
    )

print(f" Saved dataset")

print("X shape:", all_x_win.shape)
print("y shape:", all_y_win.shape)

 Saved dataset
X shape: (258893, 128, 15)
y shape: (258893, 128)
